# Alignment of Macroeconomic and Geopolitical Datasets

<p><i>Author:</i> Angelica Vanti</p>

<p><i>Project:</i> Predicting EUR/USD Movements Using Geopolitical and Macroeconomic Variables</p>

This notebook aligns the stationary macroeconomic and financial variables with the five alternative daily geopolitical-risk specifications constructed from LLM-scored tweets.

The objective is to create a common modelling sample in which the baseline macroeconomic specification and all geopolitical extensions contain identical macroeconomic observations and dates. This ensures that differences in subsequent forecasting performance are attributable to the inclusion and representation of geopolitical information rather than differences in sample composition.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"

# Inputs
MACRO_PATH = PROCESSED_DIR / "macro_processed.csv"

TRUTHS_MAX_PATH = PROCESSED_DIR / "truths_daily_max_deepseek.csv"
TRUTHS_SUM_PATH = PROCESSED_DIR / "truths_daily_sum_deepseek.csv"
TRUTHS_MA3_PATH = PROCESSED_DIR / "truths_daily_sum_ma3_deepseek.csv"
TRUTHS_MA5_PATH = PROCESSED_DIR / "truths_daily_sum_ma5_deepseek.csv"
TRUTHS_MA10_PATH = PROCESSED_DIR / "truths_daily_sum_ma10_deepseek.csv"

# Final output
DEEPSEEK_ALL_PATH = (
    PROCESSED_DIR / "macro_truths_all_deepseek.csv"
)

In [4]:
macro = pd.read_csv(MACRO_PATH)

truths_max = pd.read_csv(TRUTHS_MAX_PATH)
truths_sum = pd.read_csv(TRUTHS_SUM_PATH)
truths_ma3 = pd.read_csv(TRUTHS_MA3_PATH)
truths_ma5 = pd.read_csv(TRUTHS_MA5_PATH)
truths_ma10 = pd.read_csv(TRUTHS_MA10_PATH)

print("Macro:", macro.shape)
print("MAX:", truths_max.shape)
print("SUM:", truths_sum.shape)
print("MA3:", truths_ma3.shape)
print("MA5:", truths_ma5.shape)
print("MA10:", truths_ma10.shape)

Macro: (2376, 5)
MAX: (588, 4)
SUM: (588, 4)
MA3: (588, 4)
MA5: (588, 4)
MA10: (588, 4)


## 1. Data Loading and Sample Overlap

The processed macroeconomic dataset and the five geopolitical index specifications are loaded and their date variables converted to a common datetime format.

The geopolitical sample ends on 8 January 2021. The macroeconomic dataset extends beyond this period, so only macro observations up to the end of the geopolitical sample are retained for the alignment procedure.

In [5]:
# Convert all date columns to datetime
macro["observation_date"] = pd.to_datetime(
    macro["observation_date"],
    errors="coerce"
)

for df in [truths_max, truths_sum, truths_ma3, truths_ma5, truths_ma10]:
    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

# Validate dates
assert macro["observation_date"].notna().all()

for name, df in {
    "MAX": truths_max,
    "SUM": truths_sum,
    "MA3": truths_ma3,
    "MA5": truths_ma5,
    "MA10": truths_ma10,
}.items():
    assert df["date"].notna().all(), f"{name} contains invalid dates."

# End of geopolitical sample
overlap_end = min(
    macro["observation_date"].max(),
    truths_sum["date"].max()
)

# Macro observations available during geopolitical sample
MACRO_COLS = [
    "DEXUSEU_logreturn",
    "DGS2_diff",
    "USEPUINDXD_diff",
    "VIXCLS",
]

macro_overlap = (
    macro.loc[
        macro["observation_date"] <= overlap_end,
        ["observation_date"] + MACRO_COLS
    ]
    .dropna()
    .sort_values("observation_date")
    .reset_index(drop=True)
)

print("Macro date range:")
print(
    macro["observation_date"].min(),
    "to",
    macro["observation_date"].max()
)

print("\nTruths date range:")
print(
    truths_sum["date"].min(),
    "to",
    truths_sum["date"].max()
)

print("\nMacro observations in geopolitical overlap:")
print(len(macro_overlap))

Macro date range:
2017-01-04 00:00:00 to 2026-07-24 00:00:00

Truths date range:
2024-11-01 00:00:00 to 2026-06-11 00:00:00

Macro observations in geopolitical overlap:
2347


In [6]:
# Available macro observation dates during the geopolitical sample
macro_dates = (
    macro_overlap[["observation_date"]]
    .sort_values("observation_date")
    .rename(columns={"observation_date": "macro_date"})
)


def map_to_next_macro_date(truths_df):
    """
    Assign each geopolitical calendar day to the next available
    macro observation date.

    Geopolitical information released during weekends, holidays,
    or other macro-data gaps is therefore incorporated into the
    next observed market date.
    """
    truths = truths_df.sort_values("date").copy()

    mapped = pd.merge_asof(
        truths,
        macro_dates,
        left_on="date",
        right_on="macro_date",
        direction="forward"
    )

    return mapped

## 2. Alignment of Daily MAX and SUM Indices

The tweet-based geopolitical indices are defined on a complete calendar-day basis, whereas the macroeconomic variables are observed only on available market/data dates.

For the daily MAX and SUM specifications, each geopolitical calendar day is therefore assigned to the next available macro observation date. Geopolitical information released during weekends, holidays or other gaps is consequently incorporated into the next observed market date.

Where multiple geopolitical calendar days map to the same macro date, SUM scores are added and MAX scores retain the largest assigned value.

In [8]:
# Test the alignment using the daily SUM series
sum_mapped = map_to_next_macro_date(truths_sum)

assert len(sum_mapped) == len(truths_sum)
assert sum_mapped["macro_date"].notna().all()

print("Original truths rows:", len(truths_sum))
print("Mapped rows:", len(sum_mapped))
print(
    "Rows with no future macro date:",
    sum_mapped["macro_date"].isna().sum()
)

print("\nExample around a weekend:")
display(
    sum_mapped[
        (sum_mapped["date"] >= "2025-11-21") &
        (sum_mapped["date"] <= "2025-11-24")
    ]
)

Original truths rows: 588
Mapped rows: 588
Rows with no future macro date: 0

Example around a weekend:


,date,trade_sum,sanctions_sum,fed_pressure_sum,macro_date
385,2025-11-21,0.0,0.0,0.0,2025-11-21
386,2025-11-22,60.0,40.0,0.0,2025-11-24
387,2025-11-23,135.0,40.0,0.0,2025-11-24
388,2025-11-24,90.0,0.0,0.0,2025-11-24


In [9]:
# Aggregate SUM and MAX onto macro dates

# Map both geopolitical specifications
sum_mapped = map_to_next_macro_date(truths_sum)
max_mapped = map_to_next_macro_date(truths_max)

# SUM:
# add all geopolitical activity assigned to each available macro date
sum_aligned = (
    sum_mapped
    .groupby("macro_date")[
        ["trade_sum", "sanctions_sum", "fed_pressure_sum"]
    ]
    .sum()
    .reset_index()
)

# MAX:
# keep the largest geopolitical score assigned to each available macro date
max_aligned = (
    max_mapped
    .groupby("macro_date")[
        ["trade_max", "sanctions_max", "fed_pressure_max"]
    ]
    .max()
    .reset_index()
)

print("Aligned SUM rows:", len(sum_aligned))
print("Aligned MAX rows:", len(max_aligned))

# Check on dates we manually saw were containing/missing truths/macros 

print("\nSUM example:")
display(
    sum_aligned[
        (sum_aligned["macro_date"] >= "2024-11-06") &
        (sum_aligned["macro_date"] <= "2024-11-10")
    ]
)

print("\nMAX example:")
display(
    max_aligned[
        (max_aligned["macro_date"] >= "2024-11-06") &
        (max_aligned["macro_date"] <= "2024-11-10")
    ]
)

Aligned SUM rows: 400
Aligned MAX rows: 400

SUM example:


,macro_date,trade_sum,sanctions_sum,fed_pressure_sum
3,2024-11-06,0.0,0.0,0.0
4,2024-11-07,0.0,0.0,0.0
5,2024-11-08,40.0,30.0,0.0



MAX example:


,macro_date,trade_max,sanctions_max,fed_pressure_max
3,2024-11-06,0.0,0.0,0.0
4,2024-11-07,0.0,0.0,0.0
5,2024-11-08,40.0,30.0,0.0


## 3. Alignment of Moving-Average Indices

The 3-, 5-, and 10-day moving-average geopolitical indices already incorporate information from preceding calendar days. These series are therefore aligned directly to matching macroeconomic observation dates rather than being carried forward again.

This avoids applying an additional persistence mechanism to variables that are already temporally smoothed.

In [10]:
# Align MA3, MA5, and MA10 series to macro dates

def align_moving_average_to_macro(truths_ma):
    """
    Align an already constructed calendar-day moving-average
    geopolitical index to available macro observation dates.

    No extra carry-forward is applied because the moving average
    already contains information from preceding calendar days.
    """
    aligned = macro_overlap[["observation_date"]].merge(
        truths_ma,
        left_on="observation_date",
        right_on="date",
        how="left"
    )

    return aligned.drop(columns="date")


ma3_aligned = align_moving_average_to_macro(truths_ma3)
ma5_aligned = align_moving_average_to_macro(truths_ma5)
ma10_aligned = align_moving_average_to_macro(truths_ma10)

print("MA3 aligned:", ma3_aligned.shape)
print("MA5 aligned:", ma5_aligned.shape)
print("MA10 aligned:", ma10_aligned.shape)

print("\nMissing values:")
print("\nMA3:")
print(ma3_aligned.isna().sum())

print("\nMA5:")
print(ma5_aligned.isna().sum())

print("\nMA10:")
print(ma10_aligned.isna().sum())



MA3 aligned: (2347, 4)
MA5 aligned: (2347, 4)
MA10 aligned: (2347, 4)

Missing values:

MA3:
observation_date           0
trade_sum_ma3           1948
sanctions_sum_ma3       1948
fed_pressure_sum_ma3    1948
dtype: int64

MA5:
observation_date           0
trade_sum_ma5           1949
sanctions_sum_ma5       1949
fed_pressure_sum_ma5    1949
dtype: int64

MA10:
observation_date            0
trade_sum_ma10           1953
sanctions_sum_ma10       1953
fed_pressure_sum_ma10    1953
dtype: int64


## 4. Construction of a Common Modelling Sample

To ensure that all subsequent model comparisons use exactly the same observations, the final sample is restricted to dates for which the longest geopolitical moving-average specification, MA(10), is fully available.

This produces a common sample from **10 January 2017 to 8 January 2021**, containing **993 macroeconomic observations**.

The macro-only baseline and all five geopolitical specifications are subsequently constructed on these identical dates.

In [11]:
# Use dates for which the longest moving average is fully available
common_dates = ma10_aligned.loc[
    ma10_aligned[
        [
            "trade_sum_ma10",
            "sanctions_sum_ma10",
            "fed_pressure_sum_ma10",
        ]
    ].notna().all(axis=1),
    "observation_date"
]

COMMON_START = common_dates.min()
COMMON_END = common_dates.max()

print("Common sample start:", COMMON_START)
print("Common sample end:", COMMON_END)
print("Common macro observations:", len(common_dates))

Common sample start: 2024-11-12 00:00:00
Common sample end: 2026-06-11 00:00:00
Common macro observations: 394


In [12]:
macro_common = (
    macro_overlap[
        macro_overlap["observation_date"].isin(common_dates)
    ]
    .sort_values("observation_date")
    .reset_index(drop=True)
)

print("Macro baseline:", macro_common.shape)
print(
    "Date range:",
    macro_common["observation_date"].min(),
    "to",
    macro_common["observation_date"].max()
)

Macro baseline: (394, 5)
Date range: 2024-11-12 00:00:00 to 2026-06-11 00:00:00


In [13]:
# MAX
macro_truths_max = (
    macro_common
    .merge(
        max_aligned,
        left_on="observation_date",
        right_on="macro_date",
        how="left"
    )
    .drop(columns="macro_date")
)

# SUM
macro_truths_sum = (
    macro_common
    .merge(
        sum_aligned,
        left_on="observation_date",
        right_on="macro_date",
        how="left"
    )
    .drop(columns="macro_date")
)

# MA(3)
macro_truths_ma3 = macro_common.merge(
    ma3_aligned,
    on="observation_date",
    how="left"
)

# MA(5)
macro_truths_ma5 = macro_common.merge(
    ma5_aligned,
    on="observation_date",
    how="left"
)

# MA(10)
macro_truths_ma10 = macro_common.merge(
    ma10_aligned,
    on="observation_date",
    how="left"
)

print("Baseline:", macro_common.shape)
print("MAX:     ", macro_truths_max.shape)
print("SUM:     ", macro_truths_sum.shape)
print("MA(3):   ", macro_truths_ma3.shape)
print("MA(5):   ", macro_truths_ma5.shape)
print("MA(10):  ", macro_truths_ma10.shape)

Baseline: (394, 5)
MAX:      (394, 8)
SUM:      (394, 8)
MA(3):    (394, 8)
MA(5):    (394, 8)
MA(10):   (394, 8)


## 5. Validation and Export

The six resulting modelling datasets are validated before export.

The checks confirm that:

- all specifications contain exactly 993 observations;
- no duplicate dates are present;
- no missing values remain;
- all six datasets use identical observation dates; and
- the four macroeconomic variables are numerically identical across every specification.

These checks ensure that subsequent differences in model performance cannot be attributed to differences in the underlying macroeconomic sample.

The validated datasets are then exported and reloaded to confirm that the saved files retain the expected dimensions and contain no missing or duplicate observations.

In [14]:
datasets = {
    "baseline": macro_common,
    "max": macro_truths_max,
    "sum": macro_truths_sum,
    "ma3": macro_truths_ma3,
    "ma5": macro_truths_ma5,
    "ma10": macro_truths_ma10,
}

MACRO_COLS = [
    "DEXUSEU_logreturn",
    "DGS2_diff",
    "USEPUINDXD_diff",
    "VIXCLS",
]

baseline_dates = macro_common["observation_date"].reset_index(drop=True)

for name, df in datasets.items():

    # Same sample size
    assert len(df) == 394, f"{name}: unexpected row count"

    # No duplicate dates
    assert not df["observation_date"].duplicated().any(), \
        f"{name}: duplicate dates found"

    # No missing values
    assert not df.isna().any().any(), \
        f"{name}: missing values found"

    # Identical dates
    assert df["observation_date"].reset_index(drop=True).equals(
        baseline_dates
    ), f"{name}: dates differ from baseline"

    # Identical macroeconomic data
    for col in MACRO_COLS:
        assert np.allclose(
            df[col].to_numpy(),
            macro_common[col].to_numpy()
        ), f"{name}: {col} differs from baseline"

print("All six modelling specifications passed validation.")

All six modelling specifications passed validation.


In [15]:
# Create master DeepSeek modelling dataset

truths_all = macro_truths_sum.copy()

truths_all = truths_all.merge(
    macro_truths_max[
        ["observation_date",
         "trade_max",
         "sanctions_max",
         "fed_pressure_max"]
    ],
    on="observation_date",
    how="inner"
)

truths_all = truths_all.merge(
    macro_truths_ma3[
        ["observation_date",
         "trade_sum_ma3",
         "sanctions_sum_ma3",
         "fed_pressure_sum_ma3"]
    ],
    on="observation_date",
    how="inner"
)

truths_all = truths_all.merge(
    macro_truths_ma5[
        ["observation_date",
         "trade_sum_ma5",
         "sanctions_sum_ma5",
         "fed_pressure_sum_ma5"]
    ],
    on="observation_date",
    how="inner"
)

truths_all = truths_all.merge(
    macro_truths_ma10[
        ["observation_date",
         "trade_sum_ma10",
         "sanctions_sum_ma10",
         "fed_pressure_sum_ma10"]
    ],
    on="observation_date",
    how="inner"
)

truths_all = (
    truths_all
    .sort_values("observation_date")
    .reset_index(drop=True)
)

print(truths_all.shape)

(394, 20)


In [17]:
# Export master DeepSeek modelling dataset

DEEPSEEK_ALL_PATH = PROCESSED_DIR / "truths_all_deepseek.csv"

# Final validation before export
assert truths_all.shape == (394, 20)
assert not truths_all.isna().any().any()
assert not truths_all["observation_date"].duplicated().any()

truths_all.to_csv(
    DEEPSEEK_ALL_PATH,
    index=False
)

print("Saved:", DEEPSEEK_ALL_PATH)
print("Shape:", truths_all.shape)
print(
    "Date range:",
    truths_all["observation_date"].min(),
    "to",
    truths_all["observation_date"].max()
)

Saved: ..\data\processed\truths_all_deepseek.csv
Shape: (394, 20)
Date range: 2024-11-12 00:00:00 to 2026-06-11 00:00:00


In [18]:
# Create macro-only baseline on the exact DeepSeek common sample
# Keep only target + macroeconomic variables
macro_cols = [
    "observation_date",
    "DEXUSEU_logreturn",
    "DGS2_diff",
    "USEPUINDXD_diff",
    "VIXCLS"
]

macro_validation_common = truths_all[macro_cols].copy()

# Sort chronologically and reset index
macro_validation_common = (
    macro_validation_common
    .sort_values("observation_date")
    .reset_index(drop=True)
)

# Basic checks
print("Macro baseline common sample shape:", macro_validation_common.shape)
print(
    "Date range:",
    macro_validation_common["observation_date"].min(),
    "to",
    macro_validation_common["observation_date"].max()
)

print("\nMissing values:")
print(macro_validation_common.isna().sum())

display(macro_validation_common.head())
display(macro_validation_common.tail())

Macro baseline common sample shape: (394, 5)
Date range: 2024-11-12 00:00:00 to 2026-06-11 00:00:00

Missing values:
observation_date     0
DEXUSEU_logreturn    0
DGS2_diff            0
USEPUINDXD_diff      0
VIXCLS               0
dtype: int64


,observation_date,DEXUSEU_logreturn,DGS2_diff,USEPUINDXD_diff,VIXCLS
0,2024-11-12,-0.010513,0.08,-83.06,14.71
1,2024-11-13,-0.002929,-0.07,-54.95,14.02
2,2024-11-14,-0.000473,0.07,-34.67,14.31
3,2024-11-15,-0.000947,-0.03,132.69,16.14
4,2024-11-18,0.003217,-0.02,194.80,15.58


,observation_date,DEXUSEU_logreturn,DGS2_diff,USEPUINDXD_diff,VIXCLS
389,2026-06-05,-0.008289,0.12,-133.75,21.51
390,2026-06-08,0.001040,-0.02,148.52,18.92
391,2026-06-09,0.000952,-0.02,-101.26,19.87
392,2026-06-10,-0.000519,0.00,-90.67,22.22
393,2026-06-11,-0.003035,-0.08,70.87,19.44


In [19]:
# Save macro-only common sample
macro_validation_common.to_csv(
    "../data/processed/macro_validation_common.csv",
    index=False
)

print("Saved: macro_validation_common.csv")

Saved: macro_validation_common.csv


## 6. Summary

This notebook constructed a common modelling sample combining the stationary macroeconomic variables with five alternative representations of the LLM-derived geopolitical-risk indices.

Daily MAX and SUM scores were mapped to the next available macroeconomic observation date, while the already-smoothed MA(3), MA(5), and MA(10) indices were aligned directly to observed macro dates.

The final sample was restricted to the period over which MA(10) was fully available, yielding 993 common observations between 10 January 2017 and 8 January 2021. A macro-only baseline and five geopolitical model datasets were then created on exactly the same dates and validated to contain identical macroeconomic observations.

These datasets form the direct inputs to the subsequent VAR-X model comparison.